# MediaVault · серверный импорт в Google Drive

Видео скачивается **на сервер Google Colab**, собирается в MP4, шифруется AES-GCM
и загружается в ваш Drive. Телефон передаёт небольшое задание; видеотрафик при
импорте не идёт через телефон, GitHub Pages или Cloudflare Worker.

1. На сайте MediaVault откройте свою библиотеку → **Импорт**. Укажите публичные
   ссылки, коллекцию и серии, нажмите **Подготовить серверный импорт → Скопировать задание**.
2. Здесь выберите **Среда выполнения → Выполнить всё**. Достаточно бесплатного CPU;
   GPU, Pro, банковская карта и Cloudflare не требуются.
3. Вставьте скопированное задание в появившееся поле. Затем войдите в аккаунт
   владельца и введите фразу библиотеки в скрытом поле `getpass`.
4. После завершения нажмите **Обновить библиотеку** на сайте.

Используется собственный разбор публичного HTML и открытые адаптеры плееров
`anime-dl-core`. API аниме-сайтов, DRM и платный доступ не используются.
Поддерживаются найденные публичные MP4/HLS и плееры Kodik, Aniboom, Sibnet,
Animedia. Если сайт отдаёт только проверку браузера или скрывает список серий,
понадобятся отдельные публичные ссылки; поддержка всех зеркал не гарантируется.

Бесплатные сессии Colab имеют переменную длительность и могут отключаться.
После каждой завершённой серии результат сохраняется в Drive. Повторный запуск
того же задания пропустит уже записанные серии. Временные файлы занимают место
только для текущей серии и удаляются после неё. Пока идёт импорт, не изменяйте
библиотеку с другого устройства. В Drive должно хватать свободного места.

**Не вставляйте пароль Google, Client Secret, токены или секретную фразу в JSON
задания или ячейки кода.** Фраза вводится только в скрытом поле при запуске и
не записывается в блокнот. Не делитесь работающей сессией и её выводом.

In [ ]:
# 1. Подготовка среды. Код загружается и проверяется до входа в Google.
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, urllib.request

SOURCE_HASHES = {'server/source_parser.py': '21af0d791c4971b18c7c773fd1494e1b4247cc5bb4451cc3f8b432d1b1b67f1c', 'server/media_download.py': 'ff1ec099cb7134c233cc127c6d71b1e6493476a24592c0bcf507595dc997030e', 'server/run_import.py': 'a5dd38d278d96627d32cf42f47890574ddb2a0e466f298c5e28f410f6631281d', 'server/vault_drive.py': 'af0b12fc6c4637a8643868463aee7e4649b753a34f70c1e9cc8c3415a80fd645', 'server/player_catalog.py': 'c32769841ba16262581af721f2a24c06af1f359e74fd26b8da85f1db96fbb9b8', 'server/player_compat.py': 'bd18fd66037d7bc9e2f30eaa34553d320f7477e07758b1a05c4e5da3766df3bc', 'server/browser_player.py': '14e802a0e237e124fdc84aa707794eaae16e50d3b242c35662daad38b9599809', 'server/public_page_data.py': '9c94065d91ee3f418b19f9de248abacf64dda79fb2414879d15ea72aa9998fa0', 'server/requirements.txt': 'b73183846196beb769b03b61c427bf549c9b7e1051523d34c36aa2a35cba9953', 'server/title_extras.py': '679bb5169223e64051163ceb14bb533ecfa7363c41dbfd6a8804fc0b02cfd743', 'server/manga_sources.py': '175aed6b75c1f0bfe25c59601fb3ddaea3de3d1c3f122758165e4c6088aebf0d', 'server/manga_novelcool.py': 'de92258a79a744d8842a1c706f53439cbd4736fcb0632d2706c62c3cf6206dd6', 'server/source_relay.py': '719f03d7956e81a2bb53cd0b4339e4ad361429ac1f4bfd5610a4a9c6aabccc1a', 'server/manga_raw_adapters.py': '9c3c012b62226861384beccdc0f11b0201c0a2aae14e501e54889984629a6152', 'server/manga_webtoons.py': 'e800693d1c28ea7fedf017dde3c2a9c981cb45a50dc73fa3608c58c85647d743', 'server/manga_catalog_adapters.py': '291c8cd89ebb9b7b5dcc8529842cc6c0514b4533d519d2471101e23a157a3164', 'server/manga_download.py': 'c5f6006cf4df1d9c47d6fb415e6792b60d25cac5196f0fc3b6462752308f3528', 'server/title_identity.py': '16dd6f720824cf6eb2511c4a9e829e8f8e553b583f07fa8eddeab83dd3ddaa28', 'server/suwayomi_bridge.py': '8ea3cd59588179d7e80ff32a551ccff72f4086b10190e523f4cc615bc8058cfb'}
BASE = "https://raw.githubusercontent.com/Onikri/MediaVault/main/"
RUNTIME = Path("/content/mediavault-server")
RUNTIME.mkdir(parents=True, exist_ok=True)
for relative, expected in SOURCE_HASHES.items():
    with urllib.request.urlopen(BASE + relative, timeout=45) as response:
        data = response.read(2 * 1024 * 1024)
    if hashlib.sha256(data).hexdigest() != expected:
        raise RuntimeError("Исходный код обновился. Откройте свежий блокнот по ссылке с сайта.")
    (RUNTIME / Path(relative).name).write_bytes(data)
subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-r", str(RUNTIME / "requirements.txt")], check=True)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
sys.path.insert(0, str(RUNTIME))
from run_import import validate_job, run_job
from vault_drive import GoogleDriveAdapter, VaultDrive, VaultError
from source_parser import SourceError
print("Среда готова. Платные ресурсы не включались.")

In [ ]:
# 2. Вставь задание кнопкой «Скопировать задание» на сайте (оно копируется одной строкой).
# Здесь нужны только ссылки и ID каталога. Фразу библиотеки сюда вводить нельзя.
job_text = input("Задание MediaVault (JSON): ").strip()
try:
    job = json.loads(job_text)
    validate_job(job)
except Exception:
    job_text = ""
    raise RuntimeError("Некорректное задание. Скопируйте его кнопкой на сайте заново.") from None
job_text = ""
print("Задание проверено. Далее откроется вход в Google.")

In [ ]:
# 3. Вход владельца, открытие библиотеки и серверный импорт.
from google.colab import auth
from getpass import getpass

auth.authenticate_user()
vault = None
phrase = None
try:
    # Адрес Google-аккаунта проверяется до любых записей в Drive.
    vault = VaultDrive(GoogleDriveAdapter.from_adc())
    phrase = getpass("Секретная фраза открытой библиотеки: ")
    try:
        vault.unlock(job["library_id"], phrase)
    except Exception:
        raise VaultError("Не удалось открыть библиотеку. Проверьте ID, аккаунт и секретную фразу.") from None
    finally:
        phrase = None
    result = run_job(job, vault)
    print("Откройте сайт и нажмите «Обновить библиотеку»:")
    print("https://onikri.github.io/MediaVault/?library=" + vault.library_id)
except (SourceError, VaultError) as error:
    print("Импорт остановлен:", str(error))
    print("Завершённые серии сохранены. После исправления причины можно запустить то же задание.")
except KeyboardInterrupt:
    print("Импорт прерван. Завершённые серии сохранены; то же задание можно продолжить.")
except Exception:
    print("Импорт остановлен из-за ошибки сети, доступа Google или квоты.")
    print("Проверьте свободное место и аккаунт. Повторите запуск; готовые серии будут пропущены.")
finally:
    phrase = None
    if vault is not None:
        vault.close()
    vault = None

### Что сохраняется

Drive содержит каталог, зашифрованные блоки по 4 МиБ и зашифрованные манифесты
восстановления. Разделы называются «Аниме», «Манга», «Ранобэ», «Книги», «Музыка»; имена тайтлов и
видео находятся внутри зашифрованного каталога, а папки тайтлов, сезонов/томов и файлов имеют случайные ID.
Совместимость с веб-приложением проверяется тестами Python ↔ Web Crypto.

Фраза и производный ключ используются в памяти текущего процесса. Авторизацией
Google управляет Colab. Завершите сессию через **Среда выполнения → Отключиться
и удалить среду выполнения**, когда импорт закончен. Во время обычного просмотра
на сайте видео, разумеется, скачивается из Drive на устройство зрителя.

[Исходный код и инструкции](https://github.com/Onikri/MediaVault)
· [Использованные открытые библиотеки](https://github.com/Onikri/MediaVault/blob/main/THIRD_PARTY.md)